# Reconstruction analysis (parallel, cached)

Notebook version of `reconAnaParallel.py`. The expensive step - walking the data
directory, parsing every `reco.sbc`/`bubble.sbc` pair, and matching bubble-finder
events to reco coordinates - runs across multiple processes, and results are
cached **per folder** in `CACHE_FILE`.

The cache is additive: each run reuses whatever's already cached for a folder and
only computes folders that are new, changed (`reco.sbc`/`bubble.sbc` mtime or size
differs from what's on record), or were cached under a different `RECOVER_PATH`.
Point `ROOT_PATH` at a growing dataset and re-running only pays for the new data -
nothing already cached gets thrown away or redone.

`FORCE_RECOMPUTE = True` ignores the cache for folders seen in this run (forcing a
fresh compute for all of them) and overwrites their entries with the fresh result;
it does not clear the whole cache file.

`reconLib.py` must sit next to this notebook - it holds the functions the worker
processes need (they have to be importable from a real module, not defined in a
notebook cell, to be picklable across processes).

In [ ]:
import sys, os
from pathlib import Path

# make sure reconLib (living next to this notebook) is importable regardless of
# the kernel's working directory
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import pickle
import multiprocessing
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import matplotlib.pyplot as plt

import reconLib

In [ ]:
# Configuration - edit these for your run

RECOVER_PATH = "path/to/recoVersion.py"   # module providing getProjMat(camNum)
ROOT_PATH = "path/to/data/root"           # directory to walk for reco.sbc/bubble.sbc pairs

# Caps how many *new* (uncached) folders get computed and added to the cache in a
# single run - it does not limit how much of ROOT_PATH gets scanned or how many
# already-cached folders get reused. This lets a huge dataset be onboarded a batch
# at a time across multiple runs instead of requiring one huge run up front.
LIMITER = 1000

# Anchored to NOTEBOOK_DIR (not a bare relative name) so the cache always lands in the
# same place next to this notebook, regardless of the directory Jupyter was launched from.
CACHE_FILE = NOTEBOOK_DIR / "reconAna_cache.pkl"   # matched-coordinate data gets cached here
FORCE_RECOMPUTE = False                            # set True to ignore the cache and redo the grab/match step

## Load the reco version's projection matrices

In [ ]:
getProjMat = reconLib.loadModule(RECOVER_PATH, "getProjMat")
projMatricies = [getProjMat(1), getProjMat(2), getProjMat(3)]
FIRST_PAIR_ONLY = reconLib.FIRST_PAIR_ONLY

## Grab (or reuse cached) matched coordinate data

This is the step that used to hang / crash on large datasets. It's now:
- algorithmically linear instead of quadratic (see `reconLib.grabCoords`)
- parallelized across folders with a process pool
- cached **per folder** on disk, so re-running this notebook only computes
  folders that are new, changed, or use a different reco version

In [ ]:
cache_path = Path(CACHE_FILE)

# cacheStore is a dict keyed by folder path; each value holds the matched data
# for that folder plus enough info (which reco version, source file fingerprints)
# to know whether it's still valid.
if cache_path.exists():
    with open(cache_path, "rb") as f:
        cacheStore = pickle.load(f)
    print(f"Loaded existing cache with {len(cacheStore)} folder(s) from {cache_path}.")
else:
    cacheStore = {}
    print(f"No cache file at {cache_path} yet - starting fresh.")

# walk the *whole* tree - this is cheap (just directory listings), and LIMITER
# does not cap it. LIMITER instead caps how many new folders get computed below.
pairPaths = []
for dirpath, dirnames, filenames in os.walk(ROOT_PATH):
    if "reco.sbc" in filenames and "bubble.sbc" in filenames:
        pairPaths.append((dirpath, os.path.join(dirpath, "reco.sbc"), os.path.join(dirpath, "bubble.sbc")))
print(f"Found {len(pairPaths)} candidate folder(s) total.")

# split into folders that can reuse a cached result vs folders that need (re)computing
cachedPaths = []
needsCompute = []
for dirpath, recoPath, bubblePath in pairPaths:
    entry = None if FORCE_RECOMPUTE else cacheStore.get(dirpath)
    if (entry is not None
            and entry.get("recover") == RECOVER_PATH
            and entry.get("reco_fp") == reconLib.file_fingerprint(recoPath)
            and entry.get("bubble_fp") == reconLib.file_fingerprint(bubblePath)):
        cachedPaths.append((dirpath, recoPath, bubblePath))
    else:
        needsCompute.append((dirpath, recoPath, bubblePath))

# only compute up to LIMITER new folders this run - anything past that waits for a future run
toProcess = needsCompute[:LIMITER]
remaining = len(needsCompute) - len(toProcess)

print(f"Reusing {len(cachedPaths)} cached folder(s); computing {len(toProcess)} new/changed folder(s) this run"
      + (f" ({remaining} more new/changed folder(s) left for a future run)." if remaining else "."))

if toProcess:
    ctx = multiprocessing.get_context("fork")
    with ProcessPoolExecutor(mp_context=ctx, initializer=reconLib._initWorker, initargs=(projMatricies,)) as pool:
        results = pool.map(reconLib.processPair, [p[1] for p in toProcess], [p[2] for p in toProcess])
    for (dirpath, recoPath, bubblePath), result in zip(toProcess, results):
        if result is None:
            cacheStore.pop(dirpath, None)   # drop any stale entry for an unreadable pair
            continue
        setsToAdd, recosToAdd = result
        cacheStore[dirpath] = {
            "recover": RECOVER_PATH,
            "reco_fp": reconLib.file_fingerprint(recoPath),
            "bubble_fp": reconLib.file_fingerprint(bubblePath),
            "sets": setsToAdd,
            "reco": recosToAdd,
        }
    with open(cache_path, "wb") as f:
        pickle.dump(cacheStore, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Cache updated -> {cache_path} ({len(cacheStore)} folder(s) total on disk).")
else:
    print("Nothing new to compute.")

# assemble this run's data, in walk order, from the (now up to date) cache
originalNewSets = []
reconCoords = []
for dirpath, recoPath, bubblePath in pairPaths:
    entry = cacheStore.get(dirpath)
    if entry is None:
        continue  # unreadable pair, skipped above
    originalNewSets.append(entry["sets"])
    reconCoords.append(entry["reco"])
count = len(originalNewSets)
print(f"Using {count} matched folder pair(s) for this run.")

## Per-camera reprojection plots (only when a single folder pair was matched)

In [ ]:
if count == 1:
    # group points by camera so each subplot only gets its own cam's points
    groups = {1: [], 2: [], 3: []}
    for curSet in originalNewSets:
        for item in curSet:
            cam = int(item[2])
            if cam in groups:
                groups[cam].append(item)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for i, cam_id in enumerate((1, 2, 3)):
        reconLib.plot_camera_subplot(axes[i], groups[cam_id], cam_id)
    plt.tight_layout()
    plt.savefig("camVisual.png")
else:
    print(f"Skipping per-camera plots (count={count}, not 1).")

In [ ]:
if count == 1:
    # compute the reprojection error distance for every point, split by camera
    dists_by_cam = {1: [], 2: [], 3: []}
    wayTooFarAway = 0
    total = 0
    for curSet in originalNewSets:
        for old, new, cam, ev in curSet:
            total += 1
            old = np.asarray(old, dtype=float)
            new = np.asarray(new, dtype=float)

            dist = np.linalg.norm(new - old)
            if dist <= 2000:
                dists_by_cam[int(cam)].append(dist)
                if dist > 150:
                    wayTooFarAway += 1
            else:
                wayTooFarAway += 1
    print(f"More than 150 pixels away reproj error:\t{wayTooFarAway}/{total}")

    all_dists = np.concatenate([np.asarray(dists_by_cam[k]) for k in (1, 2, 3)]) if any(dists_by_cam.values()) else np.array([])
    bins = np.linspace(0, all_dists.max(), 30) if all_dists.size else 30

    colors = {1: 'tab:blue', 2: 'tab:orange', 3: 'tab:green'}
    plt.figure(figsize=(8, 5))
    for cam in (1, 2, 3):
        plt.hist(dists_by_cam[cam], bins=bins, color=colors[cam], alpha=0.5,
                 label=f'cam {cam}', edgecolor='black', linewidth=0.3)
    plt.xlabel('Distance between coordinates (pixels)')
    plt.ylabel('Count')
    plt.title('Reprojection error')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.legend()
    plt.savefig("camHist.png")

## Detector bounds check

In [ ]:
xs, ys, zs, r2s = [], [], [], []

total = 0
badxy = 0
badr2 = 0
for curEv in reconCoords:
    for coord in curEv:
        x = coord[0][0]
        y = coord[0][1]
        z = coord[0][2]
        r2 = x**2 + y**2
        xs.append(x)
        ys.append(y)
        zs.append(z)
        r2s.append(r2)
        total += 1
        if z <= 25.4 * -8.75 - 10 or z >= 25.4 * (14.71887 - 15.358) + 10:
            badr2 += 1
        if r2 >= (25.4 * (4.525 + 0.2) + 10)**2:
            badxy += 1
print(f"Outside of y vs x bounds:\t{badxy}/{total}")
print(f"Outside of z bounds:\t{badr2}/{total}")

xs = np.asarray(xs, dtype=np.float32)
ys = np.asarray(ys, dtype=np.float32)
zs = np.asarray(zs, dtype=np.float32)
r2s = np.asarray(r2s, dtype=np.float32)

## Heat maps

In [ ]:
TARGET_BIN_MM = 3.0 if not FIRST_PAIR_ONLY else 5.0   # physical bin width, in mm, for every heat map axis
MAX_BINS_PER_AXIS = 2000                               # hard cap regardless of TARGET_BIN_MM

x_edges_xy = reconLib.mm_bin_edges(xs.min(), xs.max(), TARGET_BIN_MM, MAX_BINS_PER_AXIS)
y_edges = reconLib.mm_bin_edges(ys.min(), ys.max(), TARGET_BIN_MM, MAX_BINS_PER_AXIS)
z_edges = reconLib.mm_bin_edges(zs.min(), zs.max(), TARGET_BIN_MM, MAX_BINS_PER_AXIS)
x_edges_xz = x_edges_xy

countsxy = reconLib.hist2d_counts(xs, ys, x_edges_xy, y_edges)
countsxz = reconLib.hist2d_counts(xs, zs, x_edges_xz, z_edges)

# build the r2 vs z heat map, only using points near the target region
r2mask = (zs <= 50) & (r2s >= 2500)
r2s_mask = r2s[r2mask]
zs_mask = zs[r2mask]

r_mask = np.sqrt(np.clip(r2s_mask, 0, None))          # mm^2 -> mm
r_edges = reconLib.mm_bin_edges(r_mask.min(), r_mask.max(), TARGET_BIN_MM, MAX_BINS_PER_AXIS)
r2_edges = r_edges ** 2                                # mm -> mm^2 (non-uniform spacing)
z_edges_r2 = reconLib.mm_bin_edges(zs_mask.min(), zs_mask.max(), TARGET_BIN_MM, MAX_BINS_PER_AXIS)

countsr2 = reconLib.hist2d_counts(r2s_mask, zs_mask, r2_edges, z_edges_r2)

In [ ]:
# combined figure with all three heat maps (xy, xz, r2 vs z) side by side
fig = plt.figure(figsize=(8, 8), constrained_layout=True)
gs = fig.add_gridspec(nrows=3, ncols=2, height_ratios=[1, 1, 1])

ax1 = fig.add_subplot(gs[0, 0])  # y vs x
ax2 = fig.add_subplot(gs[0, 1])  # x vs z
ax3 = fig.add_subplot(gs[2, :])  # r2 vs z

for ax in (ax1, ax2, ax3):
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True)

reconLib.draw_xy_guides(ax1)
p1 = ax1.pcolormesh(x_edges_xy, y_edges, countsxy, shading="auto", cmap="viridis")
plt.colorbar(p1, label="Bubble count", ax=ax1)
ax1.set_xlabel("x (mm)")
ax1.set_ylabel("y (mm)")
ax1.set_title("y vs x")
ax1.set_xlim(-5 * 25.4, 5 * 25.4)
ax1.set_ylim(-5 * 25.4, 5 * 25.4)

reconLib.draw_xz_guides(ax2)
p2 = ax2.pcolormesh(x_edges_xz, z_edges, countsxz, shading="auto", cmap="viridis")
plt.colorbar(p2, label="Bubble count", ax=ax2)
ax2.set_xlabel("x (mm)")
ax2.set_ylabel("z (mm)")
ax2.set_title("z vs x")
ax2.set_xlim(0, 200)
ax2.set_ylim(-250, 100)

reconLib.draw_r2z_guides(ax3)
p3 = ax3.pcolormesh(r2_edges, z_edges_r2, countsr2, shading="auto", cmap="viridis")
plt.colorbar(p3, label="Bubble count", ax=ax3)
ax3.set_xlabel("r2 (mm^2)")
ax3.set_ylabel("z (mm)")
ax3.set_xlim(2500, 20000)
ax3.set_ylim(-300, 50)
ax3.set_title("r2 vs z")
ax3.set_aspect('auto')

plt.savefig("triheatSingle.png" if FIRST_PAIR_ONLY else "triheatMult.png")

In [ ]:
# standalone y vs x heat map
fig, ax = plt.subplots()
reconLib.draw_xy_guides(ax)
mesh = ax.pcolormesh(x_edges_xy, y_edges, countsxy, shading="auto", cmap="viridis")
plt.colorbar(mesh, label="Bubble count", ax=ax)
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_title("y vs x")
ax.set_xlim(-5 * 25.4, 5 * 25.4)
ax.set_ylim(-5 * 25.4, 5 * 25.4)
ax.grid(True)
plt.savefig("recoYvXSingle.png" if FIRST_PAIR_ONLY else "recoYvXMult.png")

In [ ]:
# standalone z vs x heat map
fig, ax = plt.subplots()
reconLib.draw_xz_guides(ax)
mesh = ax.pcolormesh(x_edges_xz, z_edges, countsxz, shading="auto", cmap="viridis")
plt.colorbar(mesh, label="Bubble count", ax=ax)
ax.set_xlabel("x (mm)")
ax.set_ylabel("z (mm)")
ax.set_title("z vs x")
ax.set_xlim(0, 200)
ax.set_ylim(-250, 100)
ax.grid(True)
plt.savefig("recoZvXSingle.png" if FIRST_PAIR_ONLY else "recoZvXMult.png")

In [ ]:
# standalone r2 vs z heat map
fig, ax = plt.subplots()
reconLib.draw_r2z_guides(ax)
mesh = ax.pcolormesh(r2_edges, z_edges_r2, countsr2, shading="auto", cmap="viridis")
plt.colorbar(mesh, label="Bubble count", ax=ax)
ax.set_xlabel("r2 (mm^2)")
ax.set_ylabel("z (mm)")
ax.set_xlim(2500, 20000)
ax.set_ylim(-300, 50)
ax.set_title("r2 vs z")
ax.grid(True)
plt.savefig("recoR2vZSingle.png" if FIRST_PAIR_ONLY else "recoR2vZMult.png")